# 매일 받은 것을 확인한다 — 하루치가 제대로 들어왔나

> `notebooks/01-데이터수집/02.매일받은것을확인한다.ipynb` · 2026-09-02 · 이동원

**이 노트북이 답하는 것**: *"오늘 수집이 돌았다고 하는데, 정말 들어왔나."*

수집을 매일 자동으로 돌리기 시작하면 실패는 **조용히** 옵니다. 예외가 나면 오히려
다행이고, 무서운 것은 *"성공했다고 하는데 아무것도 안 들어온 날"* 입니다.

이 노트북은 하루치가 들어왔는지를 **세 각도**로 확인합니다. 하나로는 부족하기
때문입니다 — 특히 **행 수만 세는 검증은 덮어쓰기를 못 잡습니다.**

| 각도 | 무엇을 보나 | 못 잡는 것 |
|---|---|---|
| ① 규모 | 행이 몇 개 늘었나 | 같은 키를 덮어쓴 경우 |
| ② 대장 | 무엇을 언제 받았나 (`fetch_log`) | 받았다고 기록만 남은 경우 |
| ③ 품질 | 값이 말이 되나 (`check_data.py`) | — |


## 0. 준비 — DB 를 읽기 전용으로 연다

⚠️ **읽기 전용(`mode=ro`)으로 엽니다.** 그냥 열면 노트북이 쓰기 주체가 되고,
수집 배치가 도는 중이라면 잠금을 다툽니다. 보기만 할 것이므로 그럴 이유가 없습니다.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

# 저장소 루트 기준 상대 경로로 적는다. 절대 경로를 찍으면 사람마다 다르고,
# 이 저장소는 PUBLIC 이라 남의 폴더 구조까지 커밋된다.
DB = Path("../../data/krx_cache.db")
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)


def 표(sql: str, params=()) -> pd.DataFrame:
    """SQL 을 돌려 표로 돌려준다."""
    return pd.read_sql_query(sql, con, params=params)


print(f"DB 크기: {DB.stat().st_size / 1024 / 1024:,.0f} MB")

DB 크기: 1,362 MB


## 1. 지금 어디까지 차 있나

가장 먼저 볼 것은 **각 표의 마지막 날짜**입니다. 시세와 지수는 같은 거래일까지
차 있어야 합니다 — 한쪽만 최신이면 그날 수집이 반쪽만 돈 것입니다.


In [2]:
표("""
SELECT 'daily_price' AS 표,
       COUNT(*)                AS 행수,
       COUNT(DISTINCT code)    AS 종목수,
       COUNT(DISTINCT bas_dd)  AS 거래일,
       MIN(bas_dd)             AS 시작일,
       MAX(bas_dd)             AS 최신일
FROM daily_price
UNION ALL
SELECT 'index_price',
       COUNT(*), COUNT(DISTINCT index_name), COUNT(DISTINCT bas_dd),
       MIN(bas_dd), MAX(bas_dd)
FROM index_price
""")

,표,행수,종목수,거래일,시작일,최신일
0,daily_price,9223644,3677,4102,20100104,20260901
1,index_price,196119,51,4102,20100104,20260901


## 2. 하루치가 들어왔는지 — 행 수만 세면 안 되는 이유

"어제보다 2,765행 늘었으니 됐다"는 **불완전한 검증**입니다.

`daily_price` 의 기본키는 `(bas_dd, code)` 입니다. 같은 날짜·같은 종목을 다시 넣으면
`INSERT OR REPLACE` 가 **조용히 덮어씁니다.** 행 수는 그대로인데 내용만 바뀌죠.
그래서 늘어난 행 수가 0 이어도 사고일 수 있고, 예상대로여도 안심할 수 없습니다.

**날짜별로 몇 종목이 들어왔는지**를 함께 봐야 합니다. 종목 수가 갑자기 줄면
그날 응답이 잘린 것입니다.


In [3]:
표("""
SELECT bas_dd AS 기준일, COUNT(*) AS 종목수,
       SUM(CASE WHEN close IS NULL OR close = 0 THEN 1 ELSE 0 END) AS 종가없음
FROM daily_price
WHERE bas_dd >= '20260818'
GROUP BY bas_dd
ORDER BY bas_dd DESC
""")

,기준일,종목수,종가없음
0,20260901,2765,0
1,20260831,2766,0
2,20260828,2767,0
3,20260827,2767,0
4,20260826,2767,0
5,20260825,2767,0
6,20260824,2765,0
7,20260821,2764,0
8,20260820,2763,0
9,20260819,2763,0


위에서 종목 수가 2,760 대에서 안정적이면 정상입니다. 갑자기 수백으로 떨어진 날이
있다면 그날 KRX 응답이 잘린 것이고, 다시 받아야 합니다.

`종가없음` 이 섞여 있는 것은 **오류가 아닙니다** — 거래정지 표시행은 시·고·저가가 0 으로
옵니다. 그 판단 근거는 `scripts/check_data.py` 가 설명합니다.


## 3. 0건은 휴장인가, 아직 안 올라온 것인가

이 구별이 자동 수집의 핵심입니다.

- **휴장일**은 영원히 0건입니다 → 다시 요청하면 낭비입니다
- **오늘·어제의 0건**은 *아직 안 올라온 것*일 수 있습니다 → 다시 받아야 합니다

⚠️ 실측: 장 마감은 15:30 인데 **2026-09-01 16:04 에 그날 자료를 요청하니 0행**이었습니다.
KRX 는 확정 시세를 저녁에 올립니다. 이때 0건을 "휴장"으로 굳혀 버리면 그 하루를
**영원히 잃습니다.**

그래서 `ingest/store/krx_store.py` 는 `ZERO_ROW_RETRY_DAYS = 7` 을 둡니다 —
최근 7일 안의 0건은 건너뛰지 않고 다시 확인합니다.


In [4]:
표("""
SELECT bas_dd AS 기준일, rows AS 받은행수, fetched_at AS 받은시각
FROM fetch_log
ORDER BY bas_dd DESC
LIMIT 10
""")

,기준일,받은행수,받은시각
0,20260902,0,2026-09-02T09:42:33
1,20260901,2765,2026-09-02T09:42:39
2,20260831,2766,2026-09-01T16:04:21
3,20260828,2767,2026-09-01T16:04:20
4,20260827,2767,2026-09-01T16:04:17
5,20260826,2767,2026-09-01T16:04:19
6,20260825,2767,2026-08-26T09:36:48
7,20260824,2765,2026-08-26T09:36:45
8,20260821,2764,2026-08-26T09:36:45
9,20260820,2763,2026-08-26T09:36:45


맨 윗줄이 **오늘**인데 `받은행수` 가 0 이라면, 아직 장이 안 끝났거나 KRX 가 확정
시세를 안 올린 것입니다. 이 노트북을 만든 2026-09-02 09:15 에도 그랬습니다 —
**장이 막 열린 시각**이라 당일 확정 시세가 있을 수 없습니다.

이 0건은 7일 안에 자동으로 다시 시도되므로 손으로 할 일이 없습니다.


## 4. 값이 말이 되나 — 품질 게이트

세 번째 각도입니다. 행이 들어왔고 대장에도 남았지만, **값이 이상할** 수 있습니다.

```bash
python scripts/check_data.py
```

`error` 다섯 항목이 전부 0 이어야 통과합니다. 그중 `missing_trading_days` 는
*달력에 있는데 받은 기록조차 없는 날*을 셉니다 — 앞의 ①②와 **다른 경로로** 빠진
날짜를 잡아 주므로, 셋을 함께 보는 의미가 여기 있습니다.


## 5. 리포트가 규모를 남긴다

`reports/data_quality.json` 은 **저장소에 커밋합니다.** 반면 `data/krx_cache.db` 는
KRX 이용약관 제11조 ②(제3자 제공 금지) 때문에 올리지 않습니다.

그래서 **DB 를 안 가진 팀원에게는 이 리포트가 "자료가 얼마나 있나"를 알 수 있는
유일한 경로**입니다. 그런데 예전 리포트는 검사 결과만 담고 규모를 안 남겼습니다.
이번에 `scale` 칸을 더했습니다.


In [5]:
import json

report = json.loads(Path("../../reports/data_quality.json").read_text(encoding="utf-8"))

print("생성 시각:", report["generated_at"])
print("게이트   :", report["gate"]["status"])
print()
for 이름, 값 in report.get("scale", {}).items():
    print(f"[{이름}]")
    for k, v in 값.items():
        print(f"   {k:<14} {v:,}" if isinstance(v, int) else f"   {k:<14} {v}")

생성 시각: 2026-09-02T09:53:57+09:00
게이트   : pass

[stock]
   rows           9,223,644
   codes          3,677
   trading_days   4,102
   first_date     20100104
   last_date      20260901
[index]
   rows           196,119
   indices        51
   first_date     20100104
   last_date      20260901


이제 어제 리포트와 오늘 리포트를 나란히 놓으면 **손으로 DB 를 열지 않고도**
"몇 행 늘었나"를 볼 수 있습니다. 매일 자동 수집을 붙이면 이 비교가 곧 감시입니다.

## 정리 — 매일 무엇을 보면 되나

| 순서 | 명령 | 봐야 할 것 |
|---|---|---|
| 1 | `python scripts/fetch_krx.py --days 5` | 신규 N일 · 저장 행수 |
| 2 | `python scripts/fetch_index.py --days 5` | 지수 최신일이 시세와 같은가 |
| 3 | `python scripts/check_data.py` | `error` 다섯이 전부 0 인가 |
| 4 | `reports/data_quality.json` 의 `scale` | 어제보다 늘었나 |

이 네 줄을 한 명령으로 묶는 것이 `pipelines.ingest` 이고, 다음 작업입니다.


In [6]:
con.close()
print("닫았습니다.")

닫았습니다.
